In [3]:
#Import Library
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Matplotlib:", plt.matplotlib.__version__)

Python: 3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]
Pandas: 3.0.6
Matplotlib: 3.11.2


In [2]:
# Load Model Bahasa Indonesia
import torch
import spacy_stanza

# Stanza's trusted model checkpoints use a legacy pickle format.
_original_torch_load = torch.load

def _load_stanza_checkpoint(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)

try:
    torch.load = _load_stanza_checkpoint
    nlp = spacy_stanza.load_pipeline(
        "id",
        processors="tokenize,pos,lemma,depparse",
        use_gpu=False,
    )
finally:
    torch.load = _original_torch_load

print("Model bahasa Indonesia berhasil dimuat.")

2026-10-20 19:34:12 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-10-20 19:34:14 WARNING: Language id package default expects mwt, which has been added
2026-10-20 19:34:18 INFO: Loading these models for language: id (Indonesian):
| Processor | Package      |
----------------------------
| tokenize  | gsd          |
| mwt       | gsd          |
| pos       | gsd_charlm   |
| lemma     | gsd_nocharlm |
| depparse  | gsd_charlm   |

2026-10-20 19:34:18 INFO: Using device: cpu
2026-10-20 19:34:18 INFO: Loading: tokenize
2026-10-20 19:34:18 INFO: Loading: mwt
2026-10-20 19:34:18 INFO: Loading: pos
2026-10-20 19:34:19 INFO: Loading: lemma
2026-10-20 19:34:19 INFO: Loading: depparse
2026-10-20 19:34:20 INFO: Done loading processors!


Model bahasa Indonesia berhasil dimuat.


In [ ]:
# Uji Model
teks = "Mahasiswa sedang mempelajari pemrosesan bahasa Pemrograman."
doc = nlp(teks)

# Validasi berdasarkan hasil pemrosesan
assert len(doc) > 0, "Tidak ada token yang dihasilkan."
assert any(token.pos_ for token in doc), "POS belum tersedia."
assert any(token.dep_ for token in doc), "Dependency belum tersedia."

print("Token berhasil dibuat:", len(doc))
print("POS tagging tersedia: Ya")
print("Dependency parsing tersedia: Ya")
print("\nHasil analisis:")

for token in doc:
    print(
        f"{token.text:15} | "
        f"{token.pos_:8} | "
        f"{token.lemma_:12} | "
        f"{token.dep_:10} | "
        f"Kepala: {token.head.text}"
    )

Token berhasil dibuat: 7
POS tagging tersedia: Ya
Dependency parsing tersedia: Ya

Hasil analisis:
Mahasiswa       | PROPN    | mahasiswa    | nsubj      | Kepala: mempelajari
sedang          | AUX      | sedang       | aux        | Kepala: mempelajari
mempelajari     | VERB     | pelajar      | root       | Kepala: mempelajari
pemrosesan      | NOUN     | proses       | obj        | Kepala: mempelajari
bahasa          | NOUN     | bahasa       | compound   | Kepala: pemrosesan
Pemrograman     | PROPN    | penprograman | nmod       | Kepala: bahasa
.               | PUNCT    | .            | punct      | Kepala: mempelajari


In [ ]:
# Memproses satu kalimat
sentence = "Mahasiswa belajar Natural Language Processing."
doc = nlp(sentence)

print("Kalimat:", sentence)
print("\nToken hasil analisis:")
for token in doc:
    print(f"{token.text:<15} POS={token.pos_:<10} Dependency={token.dep_}")

Kalimat: Mahasiswa belajar Natural Language Processing.

Token hasil analisis:
Mahasiswa       POS=PROPN      Dependency=nsubj
belajar         POS=VERB       Dependency=root
Natural         POS=PROPN      Dependency=obj
Language        POS=PROPN      Dependency=flat:name
Processing      POS=PROPN      Dependency=flat:name
.               POS=PUNCT      Dependency=punct


In [ ]:
# Analisis POS Token
doc = nlp("Mahasiswa belajar Natural Language Processing.")

print(f"{'Token':<15} {'POS':<10} {'Lemma'}")
print("-" * 40)
for token in doc:
    print(f"{token.text:<15} {token.pos_:<10} {token.lemma_}")

Token           POS        Lemma
----------------------------------------
Mahasiswa       PROPN      mahasiswa
belajar         VERB       belajar
Natural         PROPN      natural
Language        PROPN      language
Processing      PROPN      processing
.               PUNCT      .


In [ ]:
# Membuat tabel analisis POS
import pandas as pd

def analyze_pos(text, nlp_model=nlp):
    doc = nlp_model(str(text))

    return pd.DataFrame([{
        "Token": token.text,
        "POS": token.pos_,
        "POS Tag": token.tag_,
        "Lemma": token.lemma_,
        "Dependency": token.dep_,
        "Head": token.head.text
    } for token in doc])

data_pos = analyze_pos("Mahasiswa belajar NLP.")
display(data_pos)

,Token,POS,POS Tag,Lemma,Dependency,Head
0,Mahasiswa,PROPN,NSD,mahasiswa,nsubj,belajar
1,belajar,VERB,NSD,belajar,root,belajar
2,NLP,PROPN,X--,nlp,obj,belajar
3,.,PUNCT,Z--,.,punct,belajar


In [ ]:
# Analisis beberapa kalimat
sentences = [
    "Mahasiswa belajar NLP.",
    "Dosen mengajar Natural Language Processing.",
    "Mahasiswa membaca buku.",
    "Sistem memproses data teks.",
    "Model bahasa memprediksi kata berikutnya."
]

for sentence in sentences:
    print(f"\nKalimat: {sentence}")
    data_result = analyze_pos(sentence)
    display(data_result[["Token", "POS", "Dependency", "Head"]])


Kalimat: Mahasiswa belajar NLP.


,Token,POS,Dependency,Head
0,Mahasiswa,PROPN,nsubj,belajar
1,belajar,VERB,root,belajar
2,NLP,PROPN,obj,belajar
3,.,PUNCT,punct,belajar



Kalimat: Dosen mengajar Natural Language Processing.


,Token,POS,Dependency,Head
0,Dosen,PROPN,nsubj,mengajar
1,mengajar,VERB,root,mengajar
2,Natural,PROPN,obj,mengajar
3,Language,PROPN,flat:name,Natural
4,Processing,PROPN,flat:name,Language
5,.,PUNCT,punct,mengajar



Kalimat: Mahasiswa membaca buku.


,Token,POS,Dependency,Head
0,Mahasiswa,PROPN,nsubj,membaca
1,membaca,VERB,root,membaca
2,buku,NOUN,obj,membaca
3,.,PUNCT,punct,membaca



Kalimat: Sistem memproses data teks.


,Token,POS,Dependency,Head
0,Sistem,PROPN,nsubj,memproses
1,memproses,VERB,root,memproses
2,data,NOUN,obj,memproses
3,teks,NOUN,compound,data
4,.,PUNCT,punct,memproses



Kalimat: Model bahasa memprediksi kata berikutnya.


,Token,POS,Dependency,Head
0,Model,NOUN,nsubj,memprediksi
1,bahasa,NOUN,compound,Model
2,memprediksi,VERB,root,memprediksi
3,kata,NOUN,obj,memprediksi
4,berikutnya,ADJ,amod,kata
5,.,PUNCT,punct,memprediksi


In [ ]:
# Dependency parsing
doc = nlp("Mahasiswa belajar NLP.")

print(f"{'Token':<18} {'Dependency':<15} {'Head'}")
print("-" * 50)
for token in doc:
    print(f"{token.text:<18} {token.dep_:<15} {token.head.text}")

Token              Dependency      Head
--------------------------------------------------
Mahasiswa          nsubj           belajar
belajar            root            belajar
NLP                obj             belajar
.                  punct           belajar


In [ ]:
# Visualisasi Dependency
from spacy import displacy
from pathlib import Path

doc = nlp("Mahasiswa belajar Natural Language Processing.")
displacy.render(doc, style="dep", jupyter=True, options={"distance": 100})

# Membuat HTML dependency tree
html = displacy.render(
    doc,
    style="dep",
    page=True,
    jupyter=False
)

# Menyimpan hasil ke file HTML
html_path = Path("dependency_tree.html")
html_path.write_text(html, encoding="utf-8")

print("Diagram dependency tree berhasil disimpan!")
print("Lokasi file:", html_path.resolve())



Diagram dependency tree berhasil disimpan!
Lokasi file: D:\NLP\Modul-1\pertemuan-04\dependency_tree.html


In [ ]:
# Analisis Dataset

import pandas as pd

df = pd.read_csv("D:/NLP/Modul-1/pertemuan-03/dataset_preprocessed.csv")
text_column = "content"  # sesuaikan

# Ambil sample untuk analisis
sample = df.sample(10, random_state=42)

for _, row in sample.iterrows():
    text = str(row[text_column])[:200]  # batasi panjang
    doc = nlp(text)
    pos_counts = {}
    for token in doc:
        pos = token.pos_
        pos_counts[pos] = pos_counts.get(pos, 0) + 1
    print(f"Teks: {text[:80]}...")
    print(f"POS Distribution: {pos_counts}")
    print()

Teks: Sangat membantu dalam menyimpan uang Dhe bets...
POS Distribution: {'ADV': 1, 'VERB': 2, 'ADP': 1, 'NOUN': 1, 'PROPN': 2}

Teks: ok...
POS Distribution: {'PROPN': 1}

Teks: Tidak mudah ribet...
POS Distribution: {'PART': 1, 'ADJ': 2}

Teks: sangat membantu untuk saya seorang driver...
POS Distribution: {'ADV': 1, 'VERB': 1, 'ADP': 1, 'PRON': 1, 'DET': 1, 'PROPN': 1}

Teks: Apk terbaik...
POS Distribution: {'NOUN': 1, 'ADJ': 1}

Teks: Mau tf susah, hrus upgrade dl,, kl g mau g bisa,,...
POS Distribution: {'AUX': 2, 'PROPN': 1, 'ADJ': 1, 'PUNCT': 5, 'NOUN': 4, 'X': 2, 'VERB': 1}

Teks: Good...
POS Distribution: {'PROPN': 1}

Teks: keren...
POS Distribution: {'ADJ': 1}

Teks: Pengen aja kasih bintang 5 mau tau nanti dapat apa dari server...
POS Distribution: {'ADV': 2, 'VERB': 2, 'NOUN': 3, 'NUM': 1, 'AUX': 2, 'PRON': 1, 'ADP': 1}

Teks: Pembayaran tagihan kredit via tokopedia kenapa sekarang ga bisa pakai gopay, apa...
POS Distribution: {'NOUN': 7, 'PROPN': 1, 'ADV': 4, 'PART': 1, 

In [ ]:
# Analisis Kesalahan
informal_texts = [
    "aku lg belajar NLP nih",
    "gue udah baca buku itu",
    "modelnya bagus bgt sih",
    "nggak ngerti sama sekali",
    "udah instal belum?"
]

print("Analisis teks informal")
print("=" * 70)
for text in informal_texts:
    doc = nlp(text)
    print(f"\nTeks: {text}")
    for token in doc:
        print(
            f"  {token.text:<15} POS={token.pos_:<10} "
            f"Dependency={token.dep_:<12} Head={token.head.text}"
        )

Analisis teks informal

Teks: aku lg belajar NLP nih
  aku             POS=PRON       Dependency=nsubj        Head=lg
  lg              POS=VERB       Dependency=root         Head=lg
  belajar         POS=VERB       Dependency=xcomp        Head=lg
  NLP             POS=PROPN      Dependency=obj          Head=belajar
  nih             POS=NOUN       Dependency=compound     Head=NLP

Teks: gue udah baca buku itu
  gue             POS=PROPN      Dependency=nsubj        Head=baca
  udah            POS=ADV        Dependency=advmod       Head=baca
  baca            POS=VERB       Dependency=root         Head=baca
  buku            POS=NOUN       Dependency=obj          Head=baca
  itu             POS=DET        Dependency=det          Head=buku

Teks: modelnya bagus bgt sih
  model           POS=NOUN       Dependency=root         Head=model
  nya             POS=PRON       Dependency=nmod:poss    Head=model
  bagus           POS=ADJ        Dependency=amod         Head=model
  bgt            

In [ ]:
# Tugas Analisis Linguistik

hasil_ringkasan = []

# Pastikan sample berisi 10 teks dari kolom content
sample_teks = (
    df["content"]
    .dropna()
    .astype(str)
    .sample(n=10, random_state=42)
    .reset_index(drop=True)
)

for nomor, text in enumerate(sample_teks, start=1):
    text = text[:200]
    df_result = analyze_pos(text, nlp)

    hasil_ringkasan.append({
        "No": nomor,
        "Kalimat": text,
        "Token": ", ".join(
            df_result["Token"].astype(str)
        ),
        "POS": ", ".join(
            df_result["POS"].astype(str)
        ),
        "Dependency": "; ".join(
            df_result["Token"].astype(str)
            + " → "
            + df_result["Head"].astype(str)
            + " ("
            + df_result["Dependency"].astype(str)
            + ")"
        ),
        "Kesalahan?": "Perlu pemeriksaan manual"
    })

df_tugas = pd.DataFrame(hasil_ringkasan)

# Tampilkan tabel satu kali tanpa indeks 0–9
display(df_tugas.style.hide(axis="index"))

print("Jumlah teks dianalisis:", len(df_tugas))


No,Kalimat,Token,POS,Dependency,Kesalahan?
1,Sangat membantu dalam menyimpan uang Dhe bets,"Sangat, membantu, dalam, menyimpan, uang, Dhe, bets","ADV, VERB, ADP, VERB, NOUN, PROPN, PROPN",Sangat → membantu (advmod); membantu → membantu (root); dalam → menyimpan (case); menyimpan → membantu (xcomp); uang → menyimpan (obj); Dhe → uang (nmod); bets → Dhe (flat:name),Perlu pemeriksaan manual
2,ok,ok,PROPN,ok → ok (root),Perlu pemeriksaan manual
3,Tidak mudah ribet,"Tidak, mudah, ribet","PART, ADJ, ADJ",Tidak → ribet (advmod); mudah → ribet (amod); ribet → ribet (root),Perlu pemeriksaan manual
4,sangat membantu untuk saya seorang driver,"sangat, membantu, untuk, saya, seorang, driver","ADV, VERB, ADP, PRON, DET, PROPN",sangat → membantu (advmod); membantu → membantu (root); untuk → saya (case); saya → membantu (obl); seorang → driver (det); driver → saya (acl),Perlu pemeriksaan manual
5,Apk terbaik,"Apk, terbaik","NOUN, ADJ",Apk → Apk (root); terbaik → Apk (amod),Perlu pemeriksaan manual
6,"Mau tf susah, hrus upgrade dl,, kl g mau g bisa,,","Mau, tf, susah, ,, hrus, upgrade, dl, ,, ,, kl, g, mau, g, bisa, ,, ,","AUX, PROPN, ADJ, PUNCT, NOUN, NOUN, NOUN, PUNCT, PUNCT, NOUN, X, VERB, X, AUX, PUNCT, PUNCT","Mau → tf (aux); tf → tf (root); susah → tf (amod); , → hrus (punct); hrus → tf (conj); upgrade → hrus (compound); dl → upgrade (compound); , → mau (punct); , → mau (punct); kl → mau (nsubj); g → kl (goeswith); mau → tf (parataxis); g → mau (goeswith); bisa → mau (advmod:emph); , → mau (punct); , → tf (punct)",Perlu pemeriksaan manual
7,Good,Good,PROPN,Good → Good (root),Perlu pemeriksaan manual
8,keren,keren,ADJ,keren → keren (root),Perlu pemeriksaan manual
9,Pengen aja kasih bintang 5 mau tau nanti dapat apa dari server,"Pengen, aja, kasih, bintang, 5, mau, tau, nanti, dapat, apa, dari, server","ADV, VERB, NOUN, NOUN, NUM, AUX, VERB, ADV, AUX, PRON, ADP, NOUN",Pengen → aja (advmod); aja → tau (advcl); kasih → tau (nsubj); bintang → kasih (compound); 5 → bintang (nummod); mau → tau (aux); tau → tau (root); nanti → apa (advmod); dapat → apa (aux); apa → tau (obj); dari → server (case); server → apa (nmod),Perlu pemeriksaan manual
10,"Pembayaran tagihan kredit via tokopedia kenapa sekarang ga bisa pakai gopay, apa karena tokopedia sudah dijual ke tiktok. Semakin merepotkan","Pembayaran, tagihan, kredit, via, tokopedia, kenapa, sekarang, ga, bisa, pakai, gopay, ,, apa, karena, tokopedia, sudah, dijual, ke, tiktok, ., Semakin, merepotkan","NOUN, NOUN, NOUN, PROPN, NOUN, ADV, ADV, PART, AUX, VERB, NOUN, PUNCT, ADV, SCONJ, NOUN, AUX, VERB, ADP, NOUN, PUNCT, ADV, VERB","Pembayaran → pakai (nsubj); tagihan → Pembayaran (compound); kredit → tagihan (compound); via → kredit (nmod); tokopedia → via (compound); kenapa → pakai (advmod); sekarang → pakai (advmod); ga → pakai (advmod); bisa → pakai (aux); pakai → pakai (root); gopay → pakai (obj); , → dijual (punct); apa → dijual (advmod); karena → dijual (mark); tokopedia → dijual (nsubj:pass); sudah → dijual (aux); dijual → pakai (advcl); ke → tiktok (case); tiktok → dijual (obl); . → pakai (punct); Semakin → merepotkan (advmod); merepotkan → merepotkan (root)",Perlu pemeriksaan manual


Jumlah teks dianalisis: 10


In [ ]:
# visualisasi untuk 10 teks dataset
from spacy import displacy
from IPython.display import display, HTML

for nomor, text in enumerate(sample_teks, start=1):
    doc = nlp(text[:200])

    print(f"Dependency Tree Kalimat {nomor}")
    print(text[:200])

    html = displacy.render(
        doc,
        style="dep",
        jupyter=False,
        options={"distance": 100}
    )

    display(HTML(html))


Dependency Tree Kalimat 1
Sangat membantu dalam menyimpan uang Dhe bets


Dependency Tree Kalimat 2
ok


Dependency Tree Kalimat 3
Tidak mudah ribet


Dependency Tree Kalimat 4
sangat membantu untuk saya seorang driver


Dependency Tree Kalimat 5
Apk terbaik


Dependency Tree Kalimat 6
Mau tf susah, hrus upgrade dl,, kl g mau g bisa,,


Dependency Tree Kalimat 7
Good


Dependency Tree Kalimat 8
keren


Dependency Tree Kalimat 9
Pengen aja kasih bintang 5 mau tau nanti dapat apa dari server


Dependency Tree Kalimat 10
Pembayaran tagihan kredit via tokopedia kenapa sekarang ga bisa pakai gopay, apa karena tokopedia sudah dijual ke tiktok. Semakin merepotkan


# Kesimpulan
Kesimpulan dari praktik materi ini adalah bahwa linguistic processing atau pemrosesan bahasa alami merupakan tahap penting dalam memahami struktur dan makna teks secara otomatis. Dari kegiatan praktik yang dilakukan, kita dapat melihat bahwa NLP tidak hanya sekadar memisahkan kata, tetapi juga mampu mengidentifikasi fungsi kata, bentuk dasar kata, hingga hubungan antar kata dalam sebuah kalimat.